# Skin Lesion Classification (ISIC 9-Class) — Transfer Learning, Classical Classifiers & Efficiency Comparison

Dataset: [Skin Cancer ISIC (9 classes)](https://www.kaggle.com/datasets/nodoubttome/skin-cancer9-classesisic)
Reference: Ratul et al., *Skin Lesions Classification Using Deep Learning Based on Dilated Convolution* (bioRxiv, 2020)

In [1]:
!pip -q install kagglehub thop xgboost tabulate

In [2]:
import os, glob, time, copy, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
from thop import profile

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.preprocessing import label_binarize
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

## 1. Dataset

In [3]:
import kagglehub

dataset_path = kagglehub.dataset_download("nodoubttome/skin-cancer9-classesisic")
train_dir = glob.glob(os.path.join(dataset_path, "**", "Train"), recursive=True)[0]
test_dir = glob.glob(os.path.join(dataset_path, "**", "Test"), recursive=True)[0]
train_dir, test_dir

Using Colab cache for faster access to the 'skin-cancer9-classesisic' dataset.


('/kaggle/input/skin-cancer9-classesisic/Skin cancer ISIC The International Skin Imaging Collaboration/Train',
 '/kaggle/input/skin-cancer9-classesisic/Skin cancer ISIC The International Skin Imaging Collaboration/Test')

In [4]:
IMG_SIZE = 160
BATCH_SIZE = 32

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_ds_aug = datasets.ImageFolder(train_dir, transform=train_tf)
train_ds_eval = datasets.ImageFolder(train_dir, transform=eval_tf)
test_set = datasets.ImageFolder(test_dir, transform=eval_tf)

class_names = train_ds_eval.classes
num_classes = len(class_names)

indices = list(range(len(train_ds_aug)))
random.shuffle(indices)
split = int(0.85 * len(indices))
train_idx, val_idx = indices[:split], indices[split:]

train_set = Subset(train_ds_aug, train_idx)
val_set = Subset(train_ds_eval, val_idx)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

num_classes, class_names

(9,
 ['actinic keratosis',
  'basal cell carcinoma',
  'dermatofibroma',
  'melanoma',
  'nevus',
  'pigmented benign keratosis',
  'seborrheic keratosis',
  'squamous cell carcinoma',
  'vascular lesion'])

## 2. Models, Training & Evaluation Helpers

In [5]:
def build_model(name, num_classes):
    if name == "AlexNet":
        m = models.alexnet(weights=models.AlexNet_Weights.DEFAULT)
        m.classifier[6] = nn.Linear(m.classifier[6].in_features, num_classes)
    elif name == "VGG16":
        m = models.vgg16(weights=models.VGG16_Weights.DEFAULT)
        m.classifier[6] = nn.Linear(m.classifier[6].in_features, num_classes)
    elif name == "VGG19":
        m = models.vgg19(weights=models.VGG19_Weights.DEFAULT)
        m.classifier[6] = nn.Linear(m.classifier[6].in_features, num_classes)
    elif name == "ResNet18":
        m = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        m.fc = nn.Linear(m.fc.in_features, num_classes)
    elif name == "ResNet50":
        m = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        m.fc = nn.Linear(m.fc.in_features, num_classes)
    elif name == "ResNet101":
        m = models.resnet101(weights=models.ResNet101_Weights.DEFAULT)
        m.fc = nn.Linear(m.fc.in_features, num_classes)
    elif name == "DenseNet121":
        m = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)
        m.classifier = nn.Linear(m.classifier.in_features, num_classes)
    elif name == "EfficientNet-B0":
        m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, num_classes)
    return m


def train_model(model, train_loader, val_loader, epochs=6, lr=1e-4):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    for epoch in range(epochs):
        model.train()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            loss = criterion(model(x), y)
            loss.backward()
            optimizer.step()

        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device), y.to(device)
                preds = model(x).argmax(dim=1)
                correct += (preds == y).sum().item()
                total += y.size(0)
        print(f"epoch {epoch+1}/{epochs} - val_acc: {correct/total:.4f}")

    return model


def evaluate_model(model, loader, num_classes):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            probs = torch.softmax(model(x), dim=1).cpu().numpy()
            all_preds.extend(probs.argmax(axis=1))
            all_labels.extend(y.numpy())
            all_probs.extend(probs)

    all_preds, all_labels, all_probs = np.array(all_preds), np.array(all_labels), np.array(all_probs)
    acc = accuracy_score(all_labels, all_preds) * 100
    prec = precision_score(all_labels, all_preds, average="macro", zero_division=0) * 100
    rec = recall_score(all_labels, all_preds, average="macro", zero_division=0) * 100
    f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0) * 100
    try:
        y_bin = label_binarize(all_labels, classes=list(range(num_classes)))
        auc = roc_auc_score(y_bin, all_probs, average="macro", multi_class="ovr") * 100
    except ValueError:
        auc = np.nan
    return acc, prec, rec, f1, auc

## 3. Table 1 — Transfer Learning Models

In [6]:
model_names = ["AlexNet", "VGG16", "VGG19", "ResNet18", "ResNet50", "ResNet101", "DenseNet121", "EfficientNet-B0"]

table1_results = {}
trained_models = {}

for name in model_names:
    print("=" * 40, name, "=" * 40)
    model = build_model(name, num_classes)
    model = train_model(model, train_loader, val_loader, epochs=6)
    table1_results[name] = evaluate_model(model, test_loader, num_classes)
    trained_models[name] = model

table1_df = pd.DataFrame(
    table1_results, index=["Accuracy (%)", "Precision (%)", "Recall (%)", "F1-Score (%)", "AUC (%)"]
).T.round(2)

table1_df

======================================== AlexNet ========================================
Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to /root/.cache/torch/hub/checkpoints/alexnet-owt-7be5be79.pth


100%|██████████| 233M/233M [00:03<00:00, 81.2MB/s]


epoch 1/6 - val_acc: 0.5952
epoch 2/6 - val_acc: 0.6518
epoch 3/6 - val_acc: 0.6548
epoch 4/6 - val_acc: 0.6815
epoch 5/6 - val_acc: 0.6667
epoch 6/6 - val_acc: 0.6458
======================================== VGG16 ========================================
Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:04<00:00, 124MB/s]


epoch 1/6 - val_acc: 0.5387
epoch 2/6 - val_acc: 0.6310
epoch 3/6 - val_acc: 0.5982
epoch 4/6 - val_acc: 0.6875
epoch 5/6 - val_acc: 0.6875
epoch 6/6 - val_acc: 0.6994
======================================== VGG19 ========================================
Downloading: "https://download.pytorch.org/models/vgg19-dcbb9e9d.pth" to /root/.cache/torch/hub/checkpoints/vgg19-dcbb9e9d.pth


...

100%|██████████| 30.8M/30.8M [00:00<00:00, 128MB/s]


epoch 1/6 - val_acc: 0.6637
epoch 2/6 - val_acc: 0.7143
epoch 3/6 - val_acc: 0.7262
epoch 4/6 - val_acc: 0.7411
epoch 5/6 - val_acc: 0.7530
epoch 6/6 - val_acc: 0.7530
======================================== EfficientNet-B0 ========================================
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 121MB/s]


epoch 1/6 - val_acc: 0.5595
epoch 2/6 - val_acc: 0.6548
epoch 3/6 - val_acc: 0.6726
epoch 4/6 - val_acc: 0.7143
epoch 5/6 - val_acc: 0.7202
epoch 6/6 - val_acc: 0.7381


,Accuracy (%),Precision (%),Recall (%),F1-Score (%),AUC (%)
AlexNet,49.15,45.88,49.31,42.83,88.68
VGG16,61.02,62.06,59.03,57.33,89.52
VGG19,49.15,55.53,49.31,42.47,90.79
ResNet18,54.24,57.64,53.47,49.48,90.52
ResNet50,53.39,57.94,52.78,51.14,89.82
ResNet101,53.39,59.05,52.78,51.42,90.64
DenseNet121,54.24,57.17,53.47,51.18,89.27
EfficientNet-B0,53.39,53.05,52.78,51.18,90.04


In [7]:
best_model_name = table1_df["Accuracy (%)"].idxmax()
best_model = trained_models[best_model_name]
print("Best model:", best_model_name)

Best model: VGG16


## 4. Table 2 — Classifiers on Deep Features from the Best Model

In [8]:
def get_feature_extractor(model, name):
    model = copy.deepcopy(model)
    if name in ["AlexNet", "VGG16", "VGG19"]:
        model.classifier[6] = nn.Identity()
    elif name in ["ResNet18", "ResNet50", "ResNet101"]:
        model.fc = nn.Identity()
    elif name == "DenseNet121":
        model.classifier = nn.Identity()
    elif name == "EfficientNet-B0":
        model.classifier[1] = nn.Identity()
    return model.to(device).eval()


def extract_features(model, loader):
    feats, labels = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            feats.append(model(x).cpu().numpy())
            labels.append(y.numpy())
    return np.concatenate(feats), np.concatenate(labels)


feat_extractor = get_feature_extractor(best_model, best_model_name)

X_train, y_train = extract_features(feat_extractor, train_loader)
X_val, y_val = extract_features(feat_extractor, val_loader)
X_test, y_test = extract_features(feat_extractor, test_loader)

X_train_full = np.concatenate([X_train, X_val])
y_train_full = np.concatenate([y_train, y_val])

X_train_full.shape, X_test.shape

((2239, 4096), (118, 4096))

In [9]:
classifiers = {
    "Logistic Regression": LogisticRegression(max_iter=2000),
    "Decision Tree": DecisionTreeClassifier(random_state=SEED),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=SEED),
    "K-Nearest Neighbors (KNN)": KNeighborsClassifier(n_neighbors=5),
    "Linear SVM": SVC(kernel="linear", probability=True, random_state=SEED),
    "RBF-SVM": SVC(kernel="rbf", probability=True, random_state=SEED),
    "XGBoost": XGBClassifier(eval_metric="mlogloss", random_state=SEED),
}

table2_results = {}
for cname, clf in classifiers.items():
    print("Training:", cname)
    clf.fit(X_train_full, y_train_full)
    preds = clf.predict(X_test)
    probs = clf.predict_proba(X_test)

    acc = accuracy_score(y_test, preds) * 100
    prec = precision_score(y_test, preds, average="macro", zero_division=0) * 100
    rec = recall_score(y_test, preds, average="macro", zero_division=0) * 100
    f1 = f1_score(y_test, preds, average="macro", zero_division=0) * 100
    try:
        y_bin = label_binarize(y_test, classes=list(range(num_classes)))
        auc = roc_auc_score(y_bin, probs, average="macro", multi_class="ovr") * 100
    except ValueError:
        auc = np.nan

    table2_results[cname] = (acc, prec, rec, f1, auc)

table2_df = pd.DataFrame(
    table2_results, index=["Accuracy (%)", "Precision (%)", "Recall (%)", "F1-Score (%)", "AUC (%)"]
).T.round(2)

table2_df

Training: Logistic Regression
Training: Decision Tree
Training: Random Forest
Training: K-Nearest Neighbors (KNN)
Training: Linear SVM
Training: RBF-SVM
Training: XGBoost


,Accuracy (%),Precision (%),Recall (%),F1-Score (%),AUC (%)
Logistic Regression,51.69,57.14,51.39,48.42,90.10
Decision Tree,49.15,50.74,49.31,47.81,71.38
Random Forest,59.32,63.75,57.64,54.84,89.19
K-Nearest Neighbors (KNN),55.08,53.82,54.17,51.52,84.71
Linear SVM,54.24,53.52,53.47,49.49,90.47
RBF-SVM,62.71,63.89,60.42,58.63,90.32
XGBoost,58.47,60.64,56.94,54.83,87.57


In [10]:
best_classifier_name = table2_df["Accuracy (%)"].idxmax()
print("Best classifier:", best_classifier_name)

Best classifier: RBF-SVM


## 5. Table 3 — Computational Efficiency Comparison

In [11]:
def count_params_m(model):
    return sum(p.numel() for p in model.parameters()) / 1e6


def get_model_size_mb(model):
    torch.save(model.state_dict(), "temp_weights.pth")
    size = os.path.getsize("temp_weights.pth") / (1024 * 1024)
    os.remove("temp_weights.pth")
    return size


def measure_inference_ms(model, runs=50):
    model.eval()
    x = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(device)
    with torch.no_grad():
        for _ in range(5):
            model(x)
        start = time.time()
        for _ in range(runs):
            model(x)
        end = time.time()
    return (end - start) / runs * 1000


table3_results = {}
for name in model_names:
    model = trained_models[name]
    x = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(device)
    flops, _ = profile(model, inputs=(x,), verbose=False)

    table3_results[name] = (
        count_params_m(model),
        get_model_size_mb(model),
        flops / 1e9,
        measure_inference_ms(model),
        table1_df.loc[name, "Accuracy (%)"],
    )

table3_df = pd.DataFrame(
    table3_results,
    index=["Parameters (M)", "Model Size (MB)", "FLOPs (G)", "Inference Time (ms)", "Accuracy (%)"],
).T.round(3)

table3_df

,Parameters (M),Model Size (MB),FLOPs (G),Inference Time (ms),Accuracy (%)
AlexNet,57.041,217.600,0.374,0.897,49.15
VGG16,134.297,512.316,7.950,6.014,61.02
VGG19,139.607,532.573,10.073,7.678,49.15
ResNet18,11.181,42.734,0.930,2.146,54.24
ResNet50,23.526,90.061,2.108,8.281,53.39
ResNet101,42.519,162.816,4.012,11.850,53.39
DenseNet121,6.963,27.183,1.478,16.888,54.24
EfficientNet-B0,4.019,15.715,0.211,9.306,53.39
